# Calidad del Aire — Ciclo estadístico completo (PM2.5, RMCAB Bogotá)

> **Bloque:** B — Transversal temática | alimenta: gestión de riesgo, sistemas de información ambiental  
> **Variable focal:** PM2.5 (µg/m³) | **Fuente:** RMCAB Bogotá / SIATA / OpenAQ  
> **Objetivo:** Ciclo estadístico completo (EDA → Descriptiva → Inferencial → Predictiva) + reporte de cumplimiento normativo

---

## Contexto institucional

Colombia monitorea la calidad del aire mediante redes regionales operadas por autoridades ambientales (CARs y corporaciones urbanas). La **RMCAB** (Red de Monitoreo de Calidad del Aire de Bogotá), operada por la **Secretaría Distrital de Ambiente (SDA)**, es la red de referencia para la Sabana de Bogotá con ~20 estaciones activas. El **SIATA** (Sistema de Alerta Temprana de Medellín y el Valle de Aburrá) opera la red antioqueña. El **IDEAM** coordina el **SISAIRE** como repositorio nacional de datos históricos.

| Actor | Rol |
|---|---|
| IDEAM | Protocolo de monitoreo, SISAIRE, datos históricos nacionales |
| SDA Bogotá | Operación RMCAB — ~20 estaciones, PM2.5/PM10/O3/NO2/CO/SO2 |
| SIATA (AMVA) | Red Antioquia-Valle de Aburrá, alertas tempranas en tiempo real |
| MinAmbiente | Regulación (Res. 2254/2017), criterios de calidad del aire |
| Alcaldías | Planes de descontaminación, Planes de Acción de Calidad del Aire |
| OpenAQ | Repositorio global open-data, incluye estaciones colombianas |

---

## Normativa colombiana aplicable

| Norma | Contaminante | Límite anual | Límite 24h | Observación |
|---|---|---|---|---|
| **Res. 2254/2017** (Colombia) | PM2.5 | 25 µg/m³ | 37 µg/m³ | Norma nacional vigente |
| **Res. 2254/2017** (Colombia) | PM10 | 50 µg/m³ | 75 µg/m³ | |
| **Res. 2254/2017** (Colombia) | O3 | — | 100 µg/m³ (8h) | |
| **Res. 2254/2017** (Colombia) | NO2 | 60 µg/m³ | 200 µg/m³ | |
| **Res. 2254/2017** (Colombia) | SO2 | 50 µg/m³ | 250 µg/m³ | |
| **Guías OMS 2021** | PM2.5 | **5 µg/m³** | **15 µg/m³** | 5× más estricta que norma CO |
| **Guías OMS 2021** | PM10 | 15 µg/m³ | 45 µg/m³ | |

### ICA Colombia — categorías (config.ICA_CATEGORIES)

| Categoría | Rango ICA | PM2.5 (µg/m³) | Acción |
|---|---|---|---|
| Buena | 0–25 | 0–12 | Sin restricciones |
| Aceptable | 26–50 | 12–37 | Grupos sensibles con precaución |
| Dañina sensibles | 51–100 | 37–55 | Evitar ejercicio intenso al aire libre |
| Dañina | 101–150 | 55–150 | Grupos sensibles: no salir |
| Muy dañina | 151–200 | 150–250 | Toda la población: limitar exposición |
| Peligrosa | >200 | >250 | Emergencia — cierre de actividades |

> **ADR-002:** Los picos de PM2.5 son señal real (incendios, quemas, Semana Santa). NO hacer clipping automático.  
> **ADR-005:** Usar `config.NORMA_CO` y `config.NORMA_OMS` — nunca hardcodear umbrales en el notebook.

## 0. Setup

Se importan todos los módulos del ciclo estadístico completo. Las constantes normativas (`NORMA_CO`, `NORMA_OMS`) vienen de `config.py` — nunca usar valores hardcodeados en el análisis.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from estadistica_ambiental.io.loaders import load_csv
from estadistica_ambiental.io.validators import validate
from estadistica_ambiental.eda.variables import classify
from estadistica_ambiental.eda.quality import assess_quality
from estadistica_ambiental.eda.profiling import run_eda
from estadistica_ambiental.eda.viz import plot_series, plot_seasonal_means, plot_correlation_heatmap
from estadistica_ambiental.preprocessing.imputation import impute
from estadistica_ambiental.descriptive.univariate import summarize
from estadistica_ambiental.descriptive.temporal import decompose_stl
from estadistica_ambiental.inference.stationarity import stationarity_report
from estadistica_ambiental.inference.trend import mann_kendall
from estadistica_ambiental.inference.intervals import exceedance_probability
from estadistica_ambiental.predictive.classical import SARIMAXModel
from estadistica_ambiental.predictive.ml import XGBoostModel, RandomForestModel
from estadistica_ambiental.evaluation.metrics import evaluate
from estadistica_ambiental.evaluation.backtesting import walk_forward, compare_backtests
from estadistica_ambiental.evaluation.comparison import rank_models
from estadistica_ambiental.reporting.forecast_report import forecast_report
from estadistica_ambiental.config import NORMA_CO, NORMA_OMS

print("Setup OK")
from estadistica_ambiental.preprocessing.air_quality import categorize_ica
from estadistica_ambiental.inference.intervals import exceedance_report
from estadistica_ambiental.reporting.compliance_report import compliance_report

## 1. Datos

**¿Qué queremos saber aquí?**
- ☐ ¿De qué estación y período provienen los datos?
- ☐ ¿Cuántos días/horas de datos hay y cuál es la resolución temporal?
- ☐ ¿Qué porcentaje de faltantes tiene PM2.5?
- ☐ ¿Están disponibles las covariables meteorológicas (temperatura, humedad, viento)?

**Fuentes reales disponibles:**

| Fuente | Cobertura | Descarga | Variables |
|---|---|---|---|
| RMCAB (SDA Bogotá) | Bogotá, ~20 estaciones | `load_rmcab()` | PM2.5, PM10, O3, NO2, CO, SO2, meteorología |
| SIATA | Medellín / Valle de Aburrá | Web scraping / API | PM2.5, PM10, temperatura, humedad |
| OpenAQ | Nacional + global | `load_openaq(location_id, parameter)` | PM2.5, PM10, O3 |
| IDEAM SISAIRE | Nacional histórico | Solicitud formal | PM2.5, PM10 |

> **Nota de dominio:** Los datos son horarios en la fuente original. Para análisis de tendencia y norma colombiana se remuestrean a **diarios** (promedio). La norma 24h de Res. 2254/2017 aplica sobre el promedio diario.

> Si no hay datos reales disponibles, se usa un dataset sintético representativo como fallback (ver celda de código).

In [ ]:
from pathlib import Path

RAW_PATH = Path("data/raw/calidad_aire_CAR_2016_2026.parquet")
STATION  = "BOGOTA RURAL - MOCHUELO"

if RAW_PATH.exists():
    raw = pd.read_parquet(RAW_PATH)
    # Columnas esperadas: fecha (datetime), estacion, pm25, [temperatura, humedad, viento]
    raw["fecha"] = pd.to_datetime(raw["fecha"])
    est = raw[raw["estacion"].str.upper().str.contains("MOCHUELO", na=False)].copy()
    est = est.sort_values("fecha")

    # Resamplear a diario (los datos son horarios)
    meteo_cols = [c for c in ["temperatura", "humedad", "viento"] if c in est.columns]
    agg = {"pm25": "mean"}
    for c in meteo_cols:
        agg[c] = "mean"
    df = (
        est.set_index("fecha")
        .resample("D")
        .agg(agg)
        .reset_index()
    )
    # Filtrar a rango con datos suficientes (2017-2026)
    df = df[(df["fecha"] >= "2017-01-01") & (df["fecha"] < "2026-01-01")]
    print(f"✅ Datos reales cargados | Estación: {STATION}")
    print(f"   {len(df):,} días | {df['fecha'].min().date()} → {df['fecha'].max().date()}")
    print(f"   Faltantes PM2.5: {df['pm25'].isna().sum()} ({df['pm25'].isna().mean()*100:.1f}%)")
else:
    print(f"⚠️  {RAW_PATH} no encontrado — usando datos sintéticos de respaldo")
    np.random.seed(42)
    n = 365 * 3
    fechas   = pd.date_range("2020-01-01", periods=n, freq="D")
    trend    = np.linspace(12, 18, n)
    seasonal = 6 * np.sin(2 * np.pi * np.arange(n) / 365 - np.pi/2)
    noise    = np.random.normal(0, 3, n)
    episodes = np.where(np.random.random(n) < 0.03, np.random.uniform(50, 120, n), 0)
    pm25     = np.clip(trend + seasonal + noise + episodes, 0, None)
    temp     = 13 + 4*np.sin(2*np.pi*np.arange(n)/365) + np.random.normal(0,1,n)
    humedad  = 75 + 10*np.sin(2*np.pi*np.arange(n)/365 + np.pi) + np.random.normal(0,5,n)
    viento   = np.abs(np.random.normal(2, 1, n))
    idx_na   = np.random.choice(n, int(n*0.05), replace=False)
    pm25[idx_na] = np.nan
    df = pd.DataFrame({"fecha": fechas, "pm25": pm25,
                       "temperatura": temp.round(1),
                       "humedad": humedad.round(1), "viento": viento.round(2)})

# Agregar columnas meteorológicas sintéticas si faltan
for col, base, amp in [("temperatura", 13, 4), ("humedad", 75, 10), ("viento", 2, 0)]:
    if col not in df.columns:
        n = len(df)
        df[col] = base + amp*np.sin(2*np.pi*np.arange(n)/365) + np.random.normal(0,1,n)

print(f"\nDataset final: {df.shape[0]:,} filas × {df.shape[1]} columnas")
df.head()

## 2. Validación y EDA

**¿Qué queremos saber aquí?**
- ☐ ¿Hay gaps temporales en la serie (días sin dato)?
- ☐ ¿PM2.5 > PM10 en algún registro? (físicamente imposible — error de sensor)
- ☐ ¿Hay valores congelados (sensor bloqueado reproduciendo el mismo valor)?
- ☐ ¿Qué distribución tienen los faltantes — MAR, MCAR o MNAR?
- ☐ ¿Qué tipo de variable es PM2.5? (continua, asimétrica positiva, con cola derecha pesada)

**Riesgos de calidad específicos de calidad del aire:**
- **Congelamiento de sensor:** detectar con `eda/quality.py → FreezeInfo`. Sensores de bajo costo muy propensos.
- **Deriva temporal:** especialmente en sensores PurpleAir/SDS011 sin mantenimiento periódico.
- **Gaps por mantenimiento programado:** patrón MAR — imputar con MICE o rolling mean según duración.
- **Picos extremos reales:** incendios forestales, quemas de caña, episodios de inversión térmica — NO eliminar (ADR-002).

In [ ]:
val = validate(df, date_col="fecha")
print(val.summary())

In [ ]:
quality = assess_quality(df, date_col="fecha")
print(quality.summary())

In [ ]:
report_path = run_eda(df, output="data/output/reports/eda_calidad_aire.html",
                      title="EDA PM2.5 RMCAB Bogotá",
                      date_col="fecha", use_ydata=False)
print(f"Reporte: {report_path}")

## 3. Visualización

**¿Qué queremos saber aquí?**
- ☐ ¿La serie tiene estacionalidad visible a lo largo del año (meses secos vs. lluviosos)?
- ☐ ¿Con qué frecuencia la línea de norma CO (37 µg/m³) y guía OMS (15 µg/m³) son superadas visualmente?
- ☐ ¿Hay episodios episódicos de contaminación extrema identificables?
- ☐ ¿Qué meses del año son sistemáticamente más contaminados?

**Contexto de interpretación:** En la Sabana de Bogotá el régimen bimodal colombiano (épocas seca: dic-feb y jun-ago; lluviosa: mar-may y sep-nov) genera dos ciclos de contaminación al año. La época seca reduce la dispersión de contaminantes. Los meses de enero y julio suelen registrar los picos más altos de PM2.5.

In [ ]:
fig = plot_series(df, "fecha", "pm25", title="PM2.5 diario — RMCAB Bogotá")
ax = fig.axes[0]
ax.axhline(NORMA_CO["pm25_24h"],  color="red",    ls="--", lw=1,
           label=f"Norma CO 24h ({NORMA_CO['pm25_24h']} µg/m³)")
ax.axhline(NORMA_OMS["pm25_24h"], color="orange", ls="--", lw=1,
           label=f"Guía OMS ({NORMA_OMS['pm25_24h']} µg/m³)")
ax.legend(fontsize=8)
plt.show()

In [ ]:
plot_seasonal_means(df, "fecha", "pm25", period="month")
plt.show()

### ICA por categoría — Distribución de días

El **ICA colombiano** (Res. 2254/2017) clasifica cada día en 6 categorías según la concentración
real de PM2.5 en µg/m³. Los colores son **oficiales y obligatorios** en reportes a CAR/MinAmbiente.

- 🟢 Buena (0–12): sin restricciones
- 🟡 Aceptable (12–37): grupos sensibles con precaución
- 🟠 Dañina sensibles (37–55): evitar ejercicio intenso
- 🔴 Dañina (55–150): grupos sensibles: no salir
- 🟣 Muy dañina (150–250): toda la población: limitar exposición
- 🟤 Peligrosa (>250): emergencia

In [ ]:
# Clasificación ICA por día (Res. 2254/2017)
pm25_nonan = df["pm25"].dropna()
cats = categorize_ica(pm25_nonan.values, pollutant="pm25")

# Colores oficiales ICA Colombia
ICA_COLORS = {
    "Buena":           "#00E400",
    "Aceptable":       "#FFFF00",
    "Dañina sensibles":"#FF7E00",
    "Dañina":          "#FF0000",
    "Muy dañina":      "#8F3F97",
    "Peligrosa":       "#7E0023",
}

import pandas as pd
cat_counts = pd.Series(cats).value_counts().reindex(list(ICA_COLORS), fill_value=0)
cat_pct = (cat_counts / len(pm25_nonan) * 100).round(1)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(cat_counts.index, cat_counts.values,
              color=[ICA_COLORS[c] for c in cat_counts.index],
              edgecolor="grey", linewidth=0.5)
for bar, (cat, pct) in zip(bars, cat_pct.items()):
    if bar.get_height() > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"{bar.get_height()} días
({pct}%)",
                ha="center", va="bottom", fontsize=8)
ax.set_title("Distribución de días por categoría ICA — PM2.5 (Res. 2254/2017)", fontweight="bold")
ax.set_ylabel("Número de días")
ax.set_xlabel("")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()
print("
Resumen ICA:")
print(cat_pct.to_string())

## 4. Estadística descriptiva

**¿Qué queremos saber aquí?**
- ☐ ¿Cuál es la mediana y media de PM2.5? ¿Dónde cae respecto a la norma CO (37 µg/m³) y OMS (15 µg/m³)?
- ☐ ¿Cuántos días al año supera el límite de 24h de la norma colombiana?
- ☐ ¿Cuántos días al año supera la guía OMS (15 µg/m³)?
- ☐ ¿Qué asimetría tiene la distribución? (esperada: lognormal o gamma — cola derecha pesada)

**Umbrales de referencia para interpretación:**

| Métrica | Valor referencia | Fuente |
|---|---|---|
| Media anual Colombia | ~25 µg/m³ | Norma Res. 2254/2017 |
| Media anual OMS | ~5 µg/m³ | Guías OMS 2021 |
| Límite 24h Colombia | 37 µg/m³ | Res. 2254/2017 |
| Límite 24h OMS | 15 µg/m³ | Guías OMS 2021 |
| Típico Bogotá (RMCAB) | 15–25 µg/m³ (media) | IDEAM / SDA |
| Episodio crítico | >55 µg/m³ | ICA "Dañina" |

In [ ]:
summarize(df)

In [ ]:
exc = exceedance_probability(df["pm25"].dropna(), threshold=NORMA_CO["pm25_24h"])
exc_oms = exceedance_probability(df["pm25"].dropna(), threshold=NORMA_OMS["pm25_24h"])
print(f"Norma CO:  {exc['n_exceedances']} días ({exc['pct_exceed']:.1f}%)")
print(f"Guía OMS:  {exc_oms['n_exceedances']} días ({exc_oms['pct_exceed']:.1f}%)")

### Reporte de excedencias normativas — `exceedance_report()`

Genera una tabla comparando la serie contra **todas las normas colombianas** aplicables
para PM2.5: Res. 2254/2017 (norma nacional) y Guías OMS 2021.

La columna **`return_period_days`** es la métrica de gestión clave (BP-7): indica cada cuántos días
en promedio se supera ese umbral — más intuitivo que el porcentaje para directivos y comunicadores.

In [ ]:
# Tabla de excedencias con período de retorno (BP-7)
tabla_exc = exceedance_report(df["pm25"].dropna(), variable="pm25")
print("Tabla de excedencias PM2.5 — todas las normas aplicables:")
tabla_exc

## 5. Inferencial

**¿Qué queremos saber aquí?**
- ☐ ¿Es la serie de PM2.5 estacionaria? (ADF + KPSS obligatorio antes de ARIMA — ADR-004)
- ☐ ¿Existe tendencia significativa de aumento o reducción de PM2.5 en el período analizado?
- ☐ ¿Cuál es la tasa de cambio en µg/m³/año (Sen's slope)?
- ☐ ¿Hay un punto de cambio estructural en la serie? (Pettitt test — ej. cuarentena COVID-19 2020)

**Guía de interpretación — estacionariedad (ADR-004):**

| ADF p-valor | KPSS p-valor | Conclusión | Acción |
|---|---|---|---|
| < 0.05 | > 0.05 | Estacionaria ✅ | Usar ARIMA sin diferenciación |
| > 0.05 | < 0.05 | No estacionaria | Diferenciar d=1 |
| < 0.05 | < 0.05 | Resultado mixto | Diferenciación conservadora |
| > 0.05 | > 0.05 | Resultado mixto | Ampliar muestra |

**Guía de interpretación — Mann-Kendall:**
- **Tendencia creciente (p < 0.05):** deterioro de la calidad del aire — alertar a la autoridad ambiental.
- **Tendencia decreciente (p < 0.05):** mejora — evaluar si corresponde a medidas de control implementadas.
- **Sen's slope:** expresa la tasa en µg/m³/año. Un valor de +1 µg/m³/año es epidemiológicamente relevante en horizontes decenales.

**Nota sobre STL (descomposición):** El período natural es 365 días para datos diarios. El componente de tendencia del STL es independiente del Mann-Kendall pero complementario — permite visualizar la evolución suavizada.

In [ ]:
pm25_clean = df.set_index("fecha")["pm25"].dropna()
stationarity_report(pm25_clean)

In [ ]:
mk = mann_kendall(pm25_clean)
print(f"Tendencia: {mk['trend']} | p={mk['pval']:.4f} | slope={mk['slope']:.4f} µg/m³/día")

In [ ]:
stl = decompose_stl(pm25_clean, period=365)
fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
for ax, col, lbl in zip(axes,
    ["observed","trend","seasonal","residual"],
    ["Observado","Tendencia","Estacionalidad","Residuo"]):
    ax.plot(stl[col], lw=1, color="#1a5276")
    ax.set_ylabel(lbl, fontsize=8)
    ax.grid(alpha=0.3)
fig.suptitle("STL — PM2.5", fontweight="bold")
plt.tight_layout(); plt.show()

## 6. Preprocesamiento

**¿Qué queremos saber aquí?**
- ☐ ¿Qué método de imputación corresponde según el patrón de faltantes?
- ☐ ¿Se preservan los picos extremos reales durante la imputación?

**Guía de imputación por patrón de ausencia (calidad del aire):**

| Tipo de gap | Estrategia recomendada | Módulo |
|---|---|---|
| ≤ 3h / ≤ 3 días | Interpolación lineal | `impute(method="linear")` |
| Medianos (4–10 días) | Rolling mean ventana 24h | `impute(method="rolling")` |
| Largos (> 10 días) con meteorología disponible | MICE multivariado | `impute(method="mice")` |
| Gaps extremos (> 40%) | BRITS / SAITS | Implementación externa |

> **ADR-002:** Imputar faltantes no significa suavizar picos. Si un gap precede/sigue a un episodio extremo documentado (incendio, quema), preservar los valores extremos adyacentes.

In [ ]:
df_imp = impute(df.copy(), cols=["pm25"], method="linear")
print(f"Faltantes antes={df.pm25.isna().sum()} | después={df_imp.pm25.isna().sum()}")

## 7. Modelos predictivos

**¿Qué queremos saber aquí?**
- ☐ ¿Qué modelo tiene menor RMSE en walk-forward validation con horizonte de 7 días?
- ☐ ¿La meteorología (temperatura, humedad, viento) mejora el pronóstico sobre SARIMA univariado?
- ☐ ¿El ranking de modelos es consistente entre splits del walk-forward?

**Modelos disponibles y su justificación:**

| Modelo | Justificación para calidad del aire | Métrica primaria |
|---|---|---|
| **SARIMAX** | Baseline de literatura. Estacionalidad semanal (s=7) + meteorología como exógenas. Interpretable. | RMSE, MAE |
| **XGBoost** | Captura no linealidades entre PM2.5 y meteorología. Lag features (1,2,3,7,14 días). | RMSE, MAE |
| **RandomForest** | Robusto a outliers. Más estable que XGBoost con datos ruidosos. | RMSE, MAE |

> **ADR-003:** No usar RMSLE en PM2.5 (siempre positivo pero con ceros posibles en zonas limpias). Usar MAE + RMSE. R² es referencial, no principal.  
> **Baseline esperado RMCAB Bogotá:** RMSE ≈ 5–8 µg/m³ para horizonte 7 días con SARIMAX.

---

**BP-1 — Walk-forward con gap obligatorio (ADR-004):**

Series con ACF alta (ACF≈0.97 en lag-1h para PM2.5) producen R² inflado sin gap.
Para datos **diarios** se usa `gap=7` (1 semana); para **horarios** `gap=24`:
```python
# Sin gap: R² inflado ~8-40 puntos (evidencia pipeline CAR 2026)
walk_forward(model, ts, horizon=7, n_splits=4, gap=7)  # ← gap obligatorio
```

**BP-3 — Score multi-métrica para calidad del aire:**

`rank_models(domain='air_quality')` usa el score validado en 34 estaciones SISAIRE:
```
score = 0.30 × RMSE_norm   # RMSE / media_estación (escala-invariante)
      + 0.10 × NRMSE
      + 0.25 × HitRate_ICA  # % días en categoría ICA correcta
      + 0.20 × F1_weighted  # F1 multi-clase ICA
      + 0.15 × Recall_gt55  # ← diferenciador clave: detectar episodios críticos
```
XGBoost detectó **15.5% de episodios >55 µg/m³** vs. **0% de SARIMA** (SISAIRE CAR 2026).

In [ ]:
ts = df_imp.set_index("fecha")["pm25"]
X  = df_imp.set_index("fecha")[["temperatura","humedad","viento"]]

models = {
    "SARIMAX":      SARIMAXModel(order=(1,1,1), seasonal_order=(1,1,1,7)),
    "XGBoost":      XGBoostModel(lags=[1,2,3,7,14]),
    "RandomForest": RandomForestModel(lags=[1,2,3,7,14]),
}

results = {}
for name, model in models.items():
    print(f"Evaluando {name}...", end=" ")
    results[name] = walk_forward(model, ts, horizon=7, n_splits=4, gap=7)  # BP-1
    print(f"RMSE={results[name]['metrics']['rmse']:.3f}")

In [ ]:
ranking = rank_models(results, domain="air_quality")
ranking[["rmse","mae","r2","score","rank"]]

## 7b. Guardrails y supuestos metodológicos
<!-- GUARDRAILS: calidad_aire -->

> **Antes de publicar resultados**, verificar que se cumplen los supuestos clave del flujo. Esta sección lista los más comunes y los específicos de la línea.

### Supuestos comunes (todas las líneas)

- **Estacionariedad (ADR-004):** ADF + KPSS deben coincidir antes de aplicar ARIMA. Si discrepan, diferenciar conservadoramente o usar modelos no-ARIMA (Prophet, ML).
- **Outliers (ADR-002):** los picos ambientales son señal real (eventos, episodios). No aplicar clipping automático — sólo `preprocessing/outliers.py` opt-in y documentado.
- **Métrica primaria (ADR-003):** RMSLE NO en variables que pueden ser negativas o cero. Usar MAE + sMAPE (o NSE / KGE en hidrología) como default.
- **Tamaño muestral mínimo:** ARIMA requiere ≥ 36 observaciones; STL anual con datos diarios, ≥ 2 ciclos completos.
- **Residuos (post-fit):** verificar normalidad (Jarque-Bera) e independencia (Ljung-Box, lag = 12). Residuos correlacionados → modelo subespecificado.
- **Walk-forward con gap (BP-1):** series con ACF ≥ 0.7 inflan R². Usar `gap ≥ horizonte` en `walk_forward()`.
- **Normas oficiales:** usar `config.NORMA_*` y `config.*_THRESHOLDS` — nunca umbrales hardcodeados en el notebook (ADR-005).

### Supuestos específicos — Calidad del aire

- **Frecuencia horaria → diaria:** la norma 24h (Res. 2254/2017) aplica al promedio diario; remuestrear antes de comparar contra umbral.
- **ML > SARIMA univariado:** RF/XGBoost con lags `[1,2,3,7,14]` son producción real (RMSE≈3.7 µg/m³, HitRate ICA≈88%). SARIMAX compite sólo con meteorología completa.
- **Recall_gt55:** evaluar la detección de episodios críticos (>55 µg/m³) — diferenciador clave (XGBoost ~15%, SARIMA ~0%).
- **Lag ENSO = 2 meses** (`ENSO_LAG_MESES['calidad_aire']`).

### Antes de presentar a la autoridad ambiental

- Reportar intervalos de confianza, no sólo el punto estimado.
- Documentar el período de los datos, los gaps y el método de imputación usado.
- Registrar decisiones metodológicas no triviales en `docs/decisiones.md` (ADRs).


## 8. Reporte final

**¿Qué queremos saber aquí?**
- ☐ ¿El mejor modelo del ranking (sección 7) mantiene su ventaja en el test set de 30 días?
- ☐ ¿Las predicciones capturan la magnitud de los episodios de alta contaminación?
- ☐ ¿El reporte HTML incluye gráficos comparativos y tabla de métricas por modelo?

> El `forecast_report()` genera un HTML autocontenido con gráficos interactivos y tabla de métricas. Es el entregable para presentar a la autoridad ambiental o al equipo técnico.

In [ ]:
train, test = ts.iloc[:-30], ts.iloc[-30:]
X_tr,  X_te = X.iloc[:-30],  X.iloc[-30:]

preds = {}
for name, model in models.items():
    model.fit(train, X_tr if name=="SARIMAX" else None)
    preds[name] = model.predict(30, X_te if name=="SARIMAX" else None)

mets = {n: evaluate(test.values, p, domain="air_quality") for n, p in preds.items()}
rpt  = forecast_report(test, preds, mets,
    output="data/output/reports/forecast_calidad_aire.html",
    title="Pronóstico PM2.5 RMCAB", variable_name="PM2.5", unit="µg/m³")
print(f"Reporte: {rpt}")

### Reporte de cumplimiento normativo HTML — `compliance_report()`

Genera un reporte HTML autocontenido para presentar a la autoridad ambiental (CAR, SDA, MinAmbiente).
Incluye tabla de excedencias, gráficos de serie con líneas normativas y semáforo ICA.

In [ ]:
# Reporte de cumplimiento normativo (HTML para CAR/MinAmbiente)
import pathlib
pathlib.Path("data/output/reports").mkdir(parents=True, exist_ok=True)

# Crear df con pm25 + fecha para el reporte
df_rep = df[["fecha", "pm25"]].copy()

rpt_html = compliance_report(
    df_rep,
    variables=["pm25"],
    linea_tematica="calidad_aire",
    output="data/output/reports/cumplimiento_pm25.html",
)
print(f"Reporte de cumplimiento: {rpt_html}")

## 9. Cómo adaptar a datos reales

Cuando tengas datos reales del proyecto, sigue estos pasos:

**Paso 1 — Conectar a la fuente**
```python
from estadistica_ambiental.io.connectors import load_rmcab, load_openaq

# Opción A: RMCAB (datos locales descargados)
df = load_rmcab("data/raw/rmcab_2020_2025.csv", station="MOCHUELO")

# Opción B: OpenAQ (API directa)
df = load_openaq(location_id=225433, parameter="pm25",
                 date_from="2020-01-01", date_to="2025-12-31")
```

**Paso 2 — Validar con rangos de dominio**
```python
from estadistica_ambiental.io.validators import validate
val = validate(df, date_col="fecha", linea_tematica="calidad_aire")
print(val.summary())  # Detecta PM2.5 > PM10, valores negativos, gaps
```

**Paso 3 — Reporte de excedencias normativas**
```python
from estadistica_ambiental.inference.intervals import exceedance_report
tabla = exceedance_report(df["pm25"], variable="pm25")
# → DataFrame con días en excedencia, % anual, comparación CO vs. OMS
```

**Paso 4 — Reporte de cumplimiento HTML para la autoridad ambiental**
```python
from estadistica_ambiental.reporting.compliance_report import compliance_report
compliance_report(df, variables=["pm25", "pm10"],
                  linea_tematica="calidad_aire",
                  output="data/output/reports/cumplimiento_rmcab.html")
```

**Paso 5 — Sustituir datos sintéticos por reales en celda 1**
- Colocar el archivo `.parquet` o `.csv` en `data/raw/`
- Ajustar `RAW_PATH` y `STATION` en la celda de carga
- El fallback sintético se desactiva automáticamente cuando el archivo existe

---

### Preguntas abiertas para explorar con datos reales
- ¿Hay diferencias significativas entre estaciones urbanas, periurbanas y rurales de la RMCAB?
- ¿La cuarentena COVID-19 (mar-sep 2020) generó un punto de cambio estructural? (Pettitt test)
- ¿El ONI/ENSO tiene correlación rezagada con PM2.5 en Bogotá? (`enso_lagged()` con lag 2 meses)
- ¿Qué sensores de bajo costo (PurpleAir) tienen mayor deriva respecto a las estaciones de referencia?

---

### Glosario mínimo
| Término | Definición |
|---|---|
| PM2.5 | Partículas ≤ 2.5 µm. Penetran alvéolos pulmonares. Principal indicador de calidad del aire interior y urbano. |
| ICA | Índice de Calidad del Aire colombiano — escala 0–500 calculado por contaminante. |
| RMCAB | Red de Monitoreo de Calidad del Aire de Bogotá — operada por la SDA. |
| SIATA | Sistema de Alerta Temprana de Medellín y el Valle de Aburrá — operado por AMVA. |
| AOD | Aerosol Optical Depth — medida satelital de columna de aerosoles (MODIS/MISR). |
| MAR | Missing At Random — faltantes explicados por otras variables observadas (ej. mantenimiento). |
| MNAR | Missing Not At Random — faltantes correlacionados con el propio valor (sensores saturados). |
| LUR | Land Use Regression — modelo de exposición a PM2.5 basado en uso del suelo urbano. |

### Referencias normativas
- Resolución 2254/2017 — Ministerio de Ambiente y Desarrollo Sostenible de Colombia
- Guías de Calidad del Aire OMS 2021 — World Health Organization  
- IDEAM. Protocolo para el Monitoreo y Seguimiento de la Calidad del Aire, 2010